# Cubism on a cube tessellation

The next piece: Tristan Miller's Cube Tessellation, folded from an 8 × 8 grid, painted as a cubist guitar.

**The geometry.** Each cube shows three faces. Tilt the model until you look straight along one cube axis, about 55° off the paper's normal, and two of the three faces turn edge-on and vanish, while the third shows as true squares. So every visible bit of paper belongs to exactly one axis.

**The piece.** Head-on is the artwork and the key: the guitar's outline is clearest there, filled with fragments. Each face is its own fragment, a piece of the guitar from its own angle: front views on the x faces, side views on y, views down the neck on z. Tilted toward any axis, the outline breaks up and you see a jumble of that axis's fragments.

**This demo is virtual:** no paper, no visibility map, and the finished model's 25 faces as ideal squares (step 18 of the diagrams), leaving out the flat triangles at its rim.

1. **The composition** (no GPU): a 3D guitar decides the outline and each face's fragment.
2. **The painting** (GPU): Stable Diffusion XL with a ControlNet paints the composition in bold colors and sharp geometric figures.

## Setup

The geometry and renderer. No GPU needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

# The finished model's faces (step 18 of the diagrams). Start from cubes at lattice points of the
# plane x + y + z = 0: head-on, along (1, 1, 1), they tile as tumbling blocks, each showing its +x, +y
# and +z faces. Nine cubes give rows of 1, 2, 3, 2, 1. The model differs at the bottom and the sides:
# - The bottom is a hollow corner, not a cube: drop the bottom cube's two side faces, and the outer
#   side faces of the two cubes above it.
# - Each side has a wall rising from the outer cube's top to the second row.
CUBES = [(a, b, -(a + b)) for a in range(3) for b in range(3)]
DROP = {(0, (3, 2, -4)), (1, (2, 3, -4)), (0, (3, 1, -3)), (1, (1, 3, -3))}   # (axis, corner)
WALLS = [(1, (2, 0, -1)), (0, (0, 2, -1))]                  # (axis, corner)
AXES = ("x", "y", "z")
E = np.eye(3)
HEAD_ON = np.ones(3) / np.sqrt(3)
HEAD_ON_UP = np.array([-1.0, -1.0, 2.0]) / np.sqrt(6)      # screen up when seen head-on
LIT = (0.8, 0.9, 1.0)                                      # x, y, z faces lit from above and to the right


def face(k, corner):
    i, j = [m for m in range(3) if m != k]
    return (k, np.array(corner, float), E[i], E[j])


def faces():
    """Every visible face: (axis it faces, corner, edge 1, edge 2)."""
    out = [face(k, np.add(p, E[k])) for p in CUBES for k in range(3)]
    out = [f for f in out if (f[0], tuple(f[1].astype(int))) not in DROP]
    return out + [face(k, c) for k, c in WALLS]


def camera(t=0.0, axis=2):
    """Toward-viewer direction and screen right and up, tilted a fraction t of the way from head-on to
    looking straight along an axis. Up stays as close to head-on's up as it can, as when tilting a
    model in your hands."""
    a, b = HEAD_ON, E[axis]
    ang = np.arccos(a @ b)
    d = (np.sin((1 - t) * ang) * a + np.sin(t * ang) * b) / np.sin(ang)
    up = HEAD_ON_UP - (HEAD_ON_UP @ d) * d
    up /= np.linalg.norm(up)
    return d, np.cross(up, d), up


def bounds(cam, fs):
    _, r, u = cam
    pts = np.array([c + s * e1 + q * e2 for _, c, e1, e2 in fs for s in (0, 1) for q in (0, 1)])
    return (pts @ r).min(), (pts @ r).max(), (pts @ u).min(), (pts @ u).max()


def render(pictures=None, t=0.0, axis=2, size=512, margin=0.06, bg=1.0, light=None, textures=None):
    """The model seen at a tilt t toward an axis. pictures: one (H, W, 3) array in [0, 1] per axis,
    each covering (center-cropped to) the box around that axis's faces as seen straight along it.
    light: a brightness per axis, to see the faces as lit paper, e.g. LIT.
    textures: instead of pictures, one (n, n, 3) array per face, rows along its edge 2."""
    fs = faces()
    d, r, u = cam = camera(t, axis)
    x0, x1, y0, y1 = bounds(cam, fs)
    span = max(x1 - x0, y1 - y0) * (1 + 2 * margin)
    X, Y = np.meshgrid((x0 + x1) / 2 + ((np.arange(size) + 0.5) / size - 0.5) * span,
                       (y0 + y1) / 2 - ((np.arange(size) + 0.5) / size - 0.5) * span)
    out = np.full((size, size, 3), bg)
    depth = np.full((size, size), -np.inf)
    cams = [camera(1.0, k) for k in range(3)]
    boxes = [bounds(cams[k], [f for f in fs if f[0] == k]) for k in range(3)]
    for i, (k, c, e1, e2) in enumerate(fs):
        if E[k] @ d < 1e-6:                                # seen edge-on
            continue
        inv = np.linalg.inv(np.array([[e1 @ r, e2 @ r], [e1 @ u, e2 @ u]]))
        qx, qy = X - c @ r, Y - c @ u
        s, q = inv[0, 0] * qx + inv[0, 1] * qy, inv[1, 0] * qx + inv[1, 1] * qy
        P = c + s[..., None] * e1 + q[..., None] * e2
        z = P @ d
        hit = (s >= 0) & (s <= 1) & (q >= 0) & (q <= 1) & (z > depth)
        if textures is not None:
            n = textures[i].shape[0]
            si, qi = (s * n).astype(int).clip(0, n - 1), (q * n).astype(int).clip(0, n - 1)
            out[hit] = textures[i][qi[hit], si[hit]] * (1 if light is None else light[k])
            depth[hit] = z[hit]
            continue
        # Each face shows the part of its axis's picture that it covers when seen along that axis.
        _, rk, uk = cams[k]
        bx0, bx1, by0, by1 = boxes[k]
        pic = pictures[k]
        H, W = pic.shape[:2]
        scale = max(W / (bx1 - bx0), H / (by1 - by0))        # cover the box, cropping the overflow
        px = ((P @ rk - (bx0 + bx1) / 2) * scale + W / 2).astype(int).clip(0, W - 1)
        py = (((by0 + by1) / 2 - P @ uk) * scale + H / 2).astype(int).clip(0, H - 1)
        out[hit] = pic[py[hit], px[hit]] * (1 if light is None else light[k])
        depth[hit] = z[hit]
    return out


def placeholder(text, color, size=256):
    """A stand-in picture: a letter on stripes, to check the geometry without a GPU."""
    img = Image.new("RGB", (size, size), color)
    draw = ImageDraw.Draw(img)
    for i in range(-size, size, 24):
        draw.line([(i, 0), (i + size, size)], fill=tuple(int(v * 0.8) for v in color), width=6)
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", size // 3)
    except OSError:
        font = ImageFont.load_default(size // 3)
    draw.text((size / 2, size / 2), text, fill="black", font=font, anchor="mm")
    return np.asarray(img, float) / 255


def plot_tilts(pictures, titles, tilts=(0, 0.25, 0.5, 0.75, 1), size=256, light=None, textures=None):
    """One row per axis: the model tilted from head-on to looking straight along that axis."""
    fig, axes = plt.subplots(3, len(tilts), figsize=(3 * len(tilts), 9.3))
    for k in range(3):
        for j, t in enumerate(tilts):
            axes[k, j].imshow(render(pictures, t, k, size, light=light, textures=textures))
            axes[k, j].set_title("head-on" if t == 0 else f"{titles[k]}" if t == 1 else f"{t:.0%} toward {AXES[k]}",
                                 fontsize=10)
            axes[k, j].axis("off")
    fig.tight_layout()
    return fig


def save_gif(pictures, path, frames=14, size=360, hold=6, light=None, textures=None):
    """Head-on, tilt to x and back, then y, then z."""
    ease = lambda s: (1 - np.cos(np.pi * s)) / 2
    imgs = []
    for k in range(3):
        ts = [ease(i / frames) for i in range(frames + 1)]
        seq = [0.0] * 2 + ts + [1.0] * hold + ts[::-1]
        imgs += [Image.fromarray((render(pictures, t, k, size, light=light, textures=textures) * 255).astype(np.uint8)) for t in seq]
    imgs[0].save(path, save_all=True, append_images=imgs[1:], duration=60, loop=0)

## The three windows

Each axis's faces form a window. Stand-in letters show the shapes and how the tilt goes: head-on (left), each cube shows a piece of all three; tilted all the way (right), the other faces have vanished edge-on.

In [ ]:
STAND_INS = [placeholder("X", (230, 120, 90)), placeholder("Y", (90, 160, 230)), placeholder("Z", (120, 200, 110))]
plot_tilts(STAND_INS, ("x", "y", "z"))
plt.show()

## The composition: a 3D guitar (no GPU)

Prompts can't reliably ask for a viewpoint, so here the viewpoints come from a 3D guitar instead, built from simple shapes. The piece works in two layers:

- **Head-on is the key: the guitar's outline.** Each face shows guitar only where a line from it straight toward the head-on viewer meets a guitar lying flat in the paper, with a dark contour at the edge. Head-on, those pieces line up into one clear guitar outline, whatever is inside them.
- **Inside, every face is its own fragment:** a close-up of one part of the guitar (soundhole, bridge, a bout, the waist, the neck or the headstock), seen roughly from the face's axis, each from its own angle, framing and roll. The fragments come from a second guitar turned so that **x sees its front, y its side, and z looks down along its neck.** Tilted toward any axis, you see a jumble of fragments: front views on x, side views on y, views down the neck on z.

This render is deliberately plain: flat tones, three tones of shading per part so each plane reads as one facet, and dark outlines. Dark gray fills where a fragment misses the guitar. It's the composition; the next section paints it with diffusion. `LIT` shades the cube faces as lit paper would be.

In [ ]:
# A guitar in its own frame, in cube edges: u along the neck toward the head, v across the body, w out
# of the top. The body is two overlapping ellipses (the bouts), 0.5 thick.
EMPTY, BODY, NECK, HEAD, BRIDGE = range(5)


def guitar(u, v, w):
    """The part at each point, or EMPTY."""
    m = np.zeros(np.shape(u), np.int8)
    lower = ((u + 1.2) / 1.0) ** 2 + (v / 1.0) ** 2 <= 1
    upper = ((u - 0.15) / 0.75) ** 2 + (v / 0.78) ** 2 <= 1
    m[(lower | upper) & (np.abs(w) <= 0.25)] = BODY
    m[(u > 0.7) & (u < 3.0) & (np.abs(v) < 0.14) & (w > -0.05) & (w < 0.25)] = NECK
    m[(u >= 3.0) & (u < 3.7) & (np.abs(v) < 0.2 + 0.1 * (u - 3)) & (w > 0.0) & (w < 0.15)] = HEAD
    m[(np.abs(u + 1.25) < 0.08) & (np.abs(v) < 0.42) & (w > 0.2) & (w < 0.32)] = BRIDGE
    return m


# Braque's palette: ochres, browns and grays.
COLORS = {"top": (0.82, 0.64, 0.38), "side": (0.56, 0.37, 0.21), "back": (0.62, 0.45, 0.28),
          "neck": (0.47, 0.31, 0.18), "fretboard": (0.2, 0.15, 0.11), "head": (0.32, 0.22, 0.13),
          "bridge": (0.2, 0.15, 0.11), "hole": (0.07, 0.06, 0.05), "rosette": (0.38, 0.3, 0.2),
          "string": (0.92, 0.9, 0.82), "background": (0.74, 0.72, 0.66), "shadow": (0.42, 0.37, 0.31)}
LIGHT = np.array([-0.3, 0.2, 1.0]) / np.linalg.norm([-0.3, 0.2, 1.0])   # from the upper left, head-on


def paint(part, u, v, w, n_world, frame):
    """The color of each hit: the part's color, the top's painted details, and flat shading in a few
    tones, so each plane reads as one facet."""
    U, V, W = frame
    out = np.zeros(part.shape + (3,))
    name = np.full(part.shape, "background", object)
    name[part == BODY] = "side"
    name[(part == BODY) & (w > 0.12)] = "top"
    name[(part == BODY) & (w < -0.12)] = "back"
    name[part == NECK] = "neck"
    name[(part == NECK) & (w > 0.12)] = "fretboard"
    name[part == HEAD] = "head"
    name[part == BRIDGE] = "bridge"
    top = name == "top"
    r = np.hypot(u + 0.3, v)
    name[top & (r < 0.36)] = "rosette"
    name[top & (r < 0.3)] = "hole"
    # Six strings from the bridge to the head, spreading toward the bridge.
    on_top = np.isin(name, ["top", "hole", "rosette", "fretboard", "bridge"]) & (w > 0.12)
    spread = np.interp(u, [-1.25, 3.0], [0.1, 0.06])
    for i in np.linspace(-1, 1, 6):
        name[on_top & (u > -1.25) & (u < 3.0) & (np.abs(v - i * spread) < 0.012)] = "string"
    for key, color in COLORS.items():
        out[name == key] = color
    shade = np.clip(n_world @ LIGHT, 0, 1)
    shade = np.round(shade * 3) / 3                   # three tones: flat facets
    hit = part != EMPTY
    out[hit] *= (0.55 + 0.45 * shade[hit])[:, None]
    return out, name


def cast(P, d, frame, center, scale, depth=8.0, steps=300):
    """Look at the guitar along direction d (toward the viewer) through a grid of points P (H, W, 3):
    each pixel shows the first part of the guitar on the line through its point. Returns a picture."""
    H, Wd = P.shape[:2]
    U, V, W = frame
    s = np.linspace(depth, -depth, steps)             # from the viewer's side inward
    first = np.zeros((H, Wd), int)
    hit = np.zeros((H, Wd), bool)
    base = P - center
    for r in range(0, H, 16):                         # a band of rows at a time, to save memory
        Q = (base[r:r + 16, :, None] + s[:, None] * d) / scale
        occupied = guitar(Q @ U, Q @ V, Q @ W) != EMPTY
        first[r:r + 16] = np.argmax(occupied, axis=2)
        hit[r:r + 16] = occupied.any(axis=2)
    # Refine each hit between the last empty step and the first full one, so flat parts come out flat.
    lo, hi = s[np.maximum(first - 1, 0)], s[first]
    for _ in range(10):
        mid = (lo + hi) / 2
        Q = (base + mid[..., None] * d) / scale
        inside = guitar(Q @ U, Q @ V, Q @ W) != EMPTY
        hi, lo = np.where(inside, mid, hi), np.where(inside, lo, mid)
    h = np.where(hit, hi, -depth)                     # height toward the viewer
    S = base + h[..., None] * d                       # the surface hit, relative to the center
    Q = S / scale
    u, v, w = Q @ U, Q @ V, Q @ W
    part = np.where(hit, guitar(u, v, w), EMPTY)
    # Normals from the surface's slope across the grid, turned toward the viewer.
    n = np.cross(np.gradient(S, axis=1), np.gradient(S, axis=0))
    n /= np.linalg.norm(n, axis=-1, keepdims=True) + 1e-12
    n *= np.sign(n @ d)[..., None]
    pic, names = paint(part, u, v, w, n, frame)
    # Outlines where the part or the height changes, like Braque's contours.
    code = np.unique(names, return_inverse=True)[1].reshape(names.shape)
    edge = np.zeros(code.shape, bool)
    for a in (0, 1):
        e = (np.diff(code, axis=a) != 0) | (np.abs(np.diff(h, axis=a)) > 0.15)
        if a == 0:
            edge[:-1] |= e; edge[1:] |= e
        else:
            edge[:, :-1] |= e; edge[:, 1:] |= e
    pic[edge] *= 0.35
    return pic


def frame_from(normal, along):
    """An orthonormal guitar frame: w along normal, u as close to along as it can."""
    W = np.asarray(normal, float); W /= np.linalg.norm(W)
    U = np.asarray(along, float) - (np.asarray(along, float) @ W) * W; U /= np.linalg.norm(U)
    return U, np.cross(W, U), W


def directions(jumble, roll, seed=0):
    """A viewing direction and a roll per face. A face of axis k looks along its axis tilted by an angle
    between jumble[k] / 2 and jumble[k] (degrees), toward a random side, and is rolled by up to roll."""
    rng = np.random.default_rng(seed)
    out = []
    for k, c, e1, e2 in faces():
        tilt = np.radians(rng.uniform(0.5, 1.0) * jumble[k])
        turn = rng.uniform(0, 2 * np.pi)
        d = np.cos(tilt) * E[k] + np.sin(tilt) * (np.cos(turn) * e1 + np.sin(turn) * e2)
        out.append((d, np.radians(rng.uniform(-roll, roll))))
    return out


def covers(P, d, frame, center, scale, depth=8.0, steps=200):
    """Whether the line through each point of P along d meets the guitar."""
    U, V, W = frame
    s = np.linspace(-depth, depth, steps)
    Q = (P[..., None, :] + s[:, None] * d - center) / scale
    return (guitar(Q @ U, Q @ V, Q @ W) != EMPTY).any(axis=-1)


# Parts of the guitar a fragment can center on, in the guitar's frame (u, v, w).
FEATURES = {"soundhole": (-0.3, 0, 0.25), "bridge": (-1.25, 0, 0.3), "lower bout": (-1.5, 0.55, 0.2),
            "upper bout": (0.2, -0.45, 0.2), "waist": (-0.55, 0.75, 0.0), "neck": (1.8, 0, 0.25),
            "headstock": (3.3, 0, 0.1)}


def cubist_textures(outline=dict(normal=HEAD_ON), outline_scale=1.15, views=dict(normal=(1, 0, 0), along=(0, -1, 2)),
                    jumble=(35, 35, 35), zoom=(0.5, 1.1), roll=25, seed=0, res=128, recipes=None):
    """What each face shows: a fragment, cut to the guitar's head-on outline.

    outline: where the guitar sits for its outline (its top facing normal, its neck toward along), at
      outline_scale, centered in the model. A face shows guitar only where a line from it straight
      toward the head-on viewer meets this guitar, with a dark contour at the edge. Head-on, the
      pieces line up into one outline.
    views: how a second guitar is turned for the fragments. The default faces its top along x and its
      neck up, so x sees its front, y its side, and z looks down along its neck.
    Each face's fragment is a close-up of one of FEATURES, chosen at random, seen along the face's own
      direction from directions(jumble, roll, seed), so roughly from its axis, framed so a face's edge
      spans between zoom[0] and zoom[1] of the guitar's units (the body is 2 across).
    recipes: a list to fill with each face's recipe (its axis, the part it shows, its angle, roll and
      framing, its window onto the second guitar, and how much of it falls inside the outline)."""
    fs = faces()
    rng = np.random.default_rng(seed + 1)
    center = np.mean([c + (e1 + e2) / 2 for _, c, e1, e2 in fs], axis=0)
    out_frame = frame_from(outline.get("normal"), outline.get("along", HEAD_ON_UP))
    view_frame = frame_from(views.get("normal"), views.get("along", HEAD_ON_UP))
    g = (np.arange(res) + 0.5) / res
    names = list(FEATURES)
    out = []
    for (k, c, e1, e2), (d, rot) in zip(fs, directions(jumble, roll, seed)):
        P = c + g[None, :, None] * e1 + g[:, None, None] * e2      # row: along e2, column: along e1
        inside = covers(P, HEAD_ON, out_frame, center, outline_scale)
        # The fragment: the face's points as seen in its axis view, laid onto a view plane across d,
        # centered on a feature of the second guitar and rolled.
        _, rk, uk = camera(1.0, k)
        x, y = (P - c - (e1 + e2) / 2) @ rk, (P - c - (e1 + e2) / 2) @ uk
        up = uk - (uk @ d) * d; up /= np.linalg.norm(up)
        right = np.cross(up, d)
        right, up = np.cos(rot) * right + np.sin(rot) * up, np.cos(rot) * up - np.sin(rot) * right
        feature = names[rng.integers(len(names))]
        f = np.array(FEATURES[feature]) @ np.array(view_frame)
        z = rng.uniform(*zoom)
        Q = f + z * (x[..., None] * right + y[..., None] * up)
        frag = cast(Q, d, view_frame, np.zeros(3), 1.0)
        if recipes is not None:
            corners = [c, c + e1, c + e1 + e2, c + e2]
            window = [(f + z * (((p - c - (e1 + e2) / 2) @ rk) * right + ((p - c - (e1 + e2) / 2) @ uk) * up)).tolist()
                      for p in corners]
            recipes.append(dict(axis=AXES[k], part=feature, tilt=float(np.degrees(np.arccos(np.clip(d @ E[k], -1, 1)))),
                                roll=float(np.degrees(rot)), zoom=float(z), direction=d.tolist(), window=window,
                                inside=float(inside.mean())))
        missed = np.all(np.abs(frag - np.array(COLORS["background"])) < 1e-6, axis=-1)
        frag[missed] = COLORS["shadow"]
        tex = np.where(inside[..., None], frag, np.array(COLORS["background"]))
        # The outline's contour.
        edge = np.zeros(inside.shape, bool)
        for a in (0, 1):
            e = np.diff(inside, axis=a)
            if a == 0:
                edge[:-1] |= e; edge[1:] |= e
            else:
                edge[:, :-1] |= e; edge[:, 1:] |= e
        tex[edge] = 0.12
        out.append(tex)
    return out

**The knobs.**

- `JUMBLE`: the most each axis's fragments tilt off their axis, in degrees (each tilts between half that and all of it, toward its own side). Higher drifts further from pure front, side and top views.
- `ZOOM`: how much of the guitar a face's edge spans, at random between the two (the body is 2 across). Lower is closer in.
- `ROLL`: the most each fragment turns in its face, in degrees.
- `SEED`: reshuffles the angles, rolls and the part each face shows.

Saved as `cubism_guitar_3d.gif`.

In [ ]:
JUMBLE = (35, 35, 35)         # x, y, z
ZOOM = (0.5, 1.1)
ROLL = 25
SEED = 0

guitar_tex = cubist_textures(jumble=JUMBLE, zoom=ZOOM, roll=ROLL, seed=SEED)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
axes[0].imshow(render(textures=guitar_tex, size=400, light=LIT)); axes[0].set_title("head-on: the outline")
for k, name in enumerate(["x: front views", "y: side views", "z: views down the neck"]):
    axes[k + 1].imshow(render(textures=guitar_tex, t=1.0, axis=k, size=300)); axes[k + 1].set_title(name)
for ax in axes:
    ax.axis("off")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(render(textures=guitar_tex, size=768, light=LIT)); ax.set_title("head-on"); ax.axis("off")
plt.show()

plot_tilts(None, ("x", "y", "z"), light=LIT, textures=guitar_tex)
plt.show()

from IPython.display import Image as ShowImage
save_gif(None, "cubism_guitar_3d.gif", light=LIT, textures=guitar_tex)
ShowImage(filename="cubism_guitar_3d.gif")

## Paint it with diffusion (needs a GPU)

The composition above fixes where things go: the guitar's outline head-on, a fragment on every face, and the cube edges. Diffusion paints it, in bold colors and sharp geometric figures, closer to Juan Gris and Léger than to Braque's browns.

1. **The control image:** the edges of the head-on render, plus every face's border, as white lines on black.
2. **The painting:** Stable Diffusion XL with a canny ControlNet paints one head-on picture that follows those edges: the outline holds, each fragment keeps its shapes, and the cube edges become hard edges in the painting.
3. **Back onto the faces:** each face takes its piece of the painting. Head-on, you see the painting. Tilted, each face's fragment moves with its face, so the tilts are still jumbles.

Use a **GPU runtime** (Runtime → Change runtime type → T4). The models are open, so no Hugging Face login is needed. Loading takes a few minutes the first time; each picture takes about a minute.

In [ ]:
import cv2

CONTROL_SIZE, CONTROL_MARGIN = 1024, 0.06


def control_image(textures, size=CONTROL_SIZE, margin=CONTROL_MARGIN):
    """White lines on black for the canny ControlNet: the edges in the head-on render (the outline's
    contour, each fragment's shapes) plus every face's border, so the painting keeps the outline,
    the fragments and the cube edges."""
    head_on = (render(textures=textures, size=size, margin=margin) * 255).astype(np.uint8)
    edges = cv2.Canny(cv2.cvtColor(head_on, cv2.COLOR_RGB2GRAY), 50, 150) > 0
    ids = [np.full((2, 2, 3), (i + 1) / 64) for i in range(len(faces()))]
    face_id = render(textures=ids, size=size, margin=margin, bg=0.0)[..., 0]
    edges[:-1] |= np.diff(face_id, axis=0) != 0
    edges[:, :-1] |= np.diff(face_id, axis=1) != 0
    edges = cv2.dilate(edges.astype(np.uint8), np.ones((2, 2), np.uint8)) > 0
    return Image.fromarray(np.where(edges, 255, 0).astype(np.uint8)).convert("RGB"), head_on


def textures_from_head_on(picture, res=128, margin=CONTROL_MARGIN):
    """Map a head-on picture (as rendered by render(t=0) at the picture's size and this margin) back
    onto the faces, as one texture per face."""
    pic = np.asarray(picture.convert("RGB") if isinstance(picture, Image.Image) else picture, float)
    pic = pic / 255 if pic.max() > 1 else pic
    size = pic.shape[0]
    fs = faces()
    _, r, u = cam = camera(0.0)
    x0, x1, y0, y1 = bounds(cam, fs)
    span = max(x1 - x0, y1 - y0) * (1 + 2 * margin)
    g = (np.arange(res) + 0.5) / res
    out = []
    for k, c, e1, e2 in fs:
        P = c + g[None, :, None] * e1 + g[:, None, None] * e2
        col = (((P @ r) - (x0 + x1) / 2) / span + 0.5) * size - 0.5
        row = (((y0 + y1) / 2 - (P @ u)) / span + 0.5) * size - 0.5
        out.append(pic[row.round().astype(int).clip(0, size - 1), col.round().astype(int).clip(0, size - 1)])
    return out

control, head_on = control_image(guitar_tex)
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(head_on); axes[0].set_title("the composition, head-on")
axes[1].imshow(control); axes[1].set_title("the control image")
for ax in axes:
    ax.axis("off")
plt.show()

In [ ]:
!pip install -q -U diffusers transformers accelerate

**The prompt.** Bold, flat, hard-edged color. `CONTROL_SCALE` is how strictly the painting follows the edges: higher keeps the outline and fragments crisper, lower gives the model more freedom. `SEEDS` makes one picture per seed.

In [ ]:
PROMPT = ("a synthetic cubist painting of a guitar, bold flat colors, sharp geometric shapes with hard straight edges, "
          "vermilion red, cobalt blue, lemon yellow, emerald green, black and white, in the style of Juan Gris and Fernand Leger")
NEGATIVE = "blurry, soft edges, gradients, photo, realistic, muddy colors, brown, beige, text, letters, watermark, signature"
CONTROL_SCALE = 0.7
GUIDANCE = 7.0
STEPS = 30
SEEDS = (0, 1, 2)

In [ ]:
import gc, torch
from diffusers import AutoencoderKL, ControlNetModel, StableDiffusionXLControlNetPipeline

if "painter" not in globals():
    # Each piece goes to the GPU as soon as it loads, so Colab's RAM never holds them all at once.
    controlnet = ControlNetModel.from_pretrained("diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16).to("cuda")
    vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16).to("cuda")
    gc.collect()
    painter = StableDiffusionXLControlNetPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0", controlnet=controlnet, vae=vae,
        torch_dtype=torch.float16, variant="fp16").to("cuda")
    gc.collect()
painter.vae.enable_tiling()       # decode in tiles, to save GPU memory

paintings = []
for seed in SEEDS:
    paintings.append(painter(PROMPT, negative_prompt=NEGATIVE, image=control, controlnet_conditioning_scale=CONTROL_SCALE,
                             guidance_scale=GUIDANCE, num_inference_steps=STEPS, width=CONTROL_SIZE, height=CONTROL_SIZE,
                             generator=torch.Generator("cuda").manual_seed(seed)).images[0])

fig, axes = plt.subplots(1, len(paintings), figsize=(6 * len(paintings), 6), squeeze=False)
for ax, seed, p in zip(axes[0], SEEDS, paintings):
    ax.imshow(p); ax.set_title(f"seed {seed}"); ax.axis("off")
plt.show()

### The painted model

Pick a painting. Head-on (lit) is the artwork; each tilt is a jumble of painted fragments. Saved as `cubism_painted.gif`.

In [ ]:
PICK = 0                      # index into SEEDS
painted_tex = textures_from_head_on(paintings[PICK])

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
axes[0].imshow(render(textures=painted_tex, size=400, light=LIT)); axes[0].set_title("head-on")
for k in range(3):
    axes[k + 1].imshow(render(textures=painted_tex, t=1.0, axis=k, size=300)); axes[k + 1].set_title(f"tilted to {AXES[k]}")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(render(textures=painted_tex, size=768, light=LIT)); ax.set_title("head-on"); ax.axis("off")
plt.show()

from IPython.display import Image as ShowImage
save_gif(None, "cubism_painted.gif", light=LIT, textures=painted_tex)
ShowImage(filename="cubism_painted.gif")

**Export for the provenance explorer.** Downloads `cubism-painting.zip`: the picked painting and the settings that made it, so the explorer can rebuild the same composition around it.

In [ ]:
import io, json, zipfile
from google.colab import files

buf = io.BytesIO()
with zipfile.ZipFile(buf, "w") as z:
    png = io.BytesIO(); paintings[PICK].save(png, "PNG")
    z.writestr("painting.png", png.getvalue())
    z.writestr("settings.json", json.dumps(dict(JUMBLE=JUMBLE, ZOOM=ZOOM, ROLL=ROLL, SEED=SEED, PROMPT=PROMPT,
        NEGATIVE=NEGATIVE, CONTROL_SCALE=CONTROL_SCALE, GUIDANCE=GUIDANCE, STEPS=STEPS, PAINT_SEED=SEEDS[PICK]), indent=1))
open("cubism-painting.zip", "wb").write(buf.getvalue())
files.download("cubism-painting.zip")

## What this leaves out

- **The real visibility map.** The demo paints ideal cube faces. The real model's faces come from scattered patches of the folded sheet, and its rim has flat triangles, so we'll need the map from sheet to faces before printing.
- **Viewing at about 55°.** That's steep, and real cubes are rounded and imperfect, so the squares won't line up exactly.
- **Diffusion sees only head-on.** The tilts get whatever the head-on painting puts on each face, stretched. A multi-view sampler could also check each tilt as it paints.